# Treinamento controlado com data augmentation

Este notebook compara `D0-A1` (geometria) e `D0-A2` (geometria + uma alteração fotométrica leve) contra o D0 já congelado. A DeepResUNet, os splits, a seed, a loss e todas as regras de avaliação permanecem iguais.

## O que está sendo testado

Cada época contém as mesmas 1.373 unidades canônicas, mas o treino recebe novos sorteios de transformações. A validação nunca recebe augmentation. A1/A2 começam do zero; nenhum checkpoint anterior é reutilizado.

## Regra de segurança

O alerta de retenção de 50% não filtra o treino. Somente uma transformação que torne vazia uma máscara originalmente positiva é repetida. Há no máximo três novos sorteios; depois disso, usa-se a amostra sem augmentation. Tudo é registrado.

## 1. Montar o Drive e localizar o projeto

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, yaml

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Execução fora do Colab.")

candidates = [
    Path("/content/drive/MyDrive/TCC/Projeto_TCC_Lucas"),
    Path("/content/drive/MyDrive/Projeto_TCC_Lucas"),
]
PROJECT_ROOT = next((p for p in candidates if p.is_dir()), Path.cwd())
SOURCE_ROOT = PROJECT_ROOT / "src"
os.chdir(PROJECT_ROOT)
os.environ["PYTHONPATH"] = str(SOURCE_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
print("Projeto:", PROJECT_ROOT)

## 2. Instalar dependências e preparar o cache temporário

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()
DATA_ROOT = Path("/content/tcc_data") if Path("/content").is_dir() else PROJECT_ROOT
if DATA_ROOT != PROJECT_ROOT:
    shutil.copytree(PROJECT_ROOT / "Dataset", DATA_ROOT / "Dataset", dirs_exist_ok=True)
print("Dados:", DATA_ROOT / "Dataset")

## 3. Conferir que somente o augmentation difere do D0

In [ ]:
from IPython.display import display

D0 = yaml.safe_load((PROJECT_ROOT / "configs/baseline_d0.yaml").read_text(encoding="utf-8"))
A1_PATH = PROJECT_ROOT / "configs/augmentation_d0_a1.yaml"
A2_PATH = PROJECT_ROOT / "configs/augmentation_d0_a2.yaml"
A1 = yaml.safe_load(A1_PATH.read_text(encoding="utf-8"))
A2 = yaml.safe_load(A2_PATH.read_text(encoding="utf-8"))
AUDIT = yaml.safe_load(
    (PROJECT_ROOT / "configs/augmentation_audit.yaml").read_text(encoding="utf-8")
)
from tcc_segmentation.augmentation_experiment import validate_augmentation_training_config

for cfg in (A1, A2):
    validate_augmentation_training_config(cfg, d0_config=D0, audit_config=AUDIT)
display(
    {
        "D0": D0["data"]["augmentation"],
        "D0-A1": A1["data"]["augmentation"],
        "D0-A2": A2["data"]["augmentation"],
        "50% filtra treino": A1["augmentation_safety"]["retention_threshold_filters_training"],
    }
)

## 4. Executar testes técnicos antes de treinar

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(SOURCE_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/test_augmentation.py",
        "tests/test_augmentation_training.py",
    ],
    cwd=PROJECT_ROOT,
    env=env,
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()

## 5. Visualizar o que a rede poderá receber

A visualização usa quatro amostras de treino e seed da época zero. Ela é um controle de implementação, não um critério para escolher A1 ou A2.

In [ ]:
import matplotlib.pyplot as plt
from tcc_segmentation.data import DevelopmentDataset
from tcc_segmentation.augmentation_experiment import make_augmentation_pair_transform

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, (label, cfg) in enumerate([("A1", A1), ("A2", A2)]):
    transform = make_augmentation_pair_transform(cfg)
    transform.set_epoch(0)
    ds = DevelopmentDataset(
        PROJECT_ROOT / cfg["data"]["train_manifest"],
        DATA_ROOT,
        "train",
        transform=transform,
        mask_threshold=cfg["data"]["mask_threshold"],
    )
    for col in range(4):
        sample = ds[col]
        axes[row, col].imshow(sample["image"].transpose(1, 2, 0))
        axes[row, col].contour(sample["mask"][0], levels=[0.5], colors=["lime"], linewidths=1)
        axes[row, col].set_title(label + " | " + sample["canonical_id"])
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

## 6. Função para acompanhar cada época

In [ ]:
def run_stream(command):
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    returncode = process.wait()
    print("\nreturncode=", returncode)
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)

# Parte A — D0-A1

Mantenha `A1_RESUME_DIR=None` para uma execução nova. Após desconexão, informe o diretório já criado para continuar a mesma execução.

In [ ]:
# Exemplo: A1_RESUME_DIR=PROJECT_ROOT/'results/day7/D0A1/D0A1_seed42_...'
A1_RESUME_DIR = None
command = [
    sys.executable,
    "scripts/run_augmentation_experiment.py",
    "--project-root",
    str(PROJECT_ROOT),
    "--data-root",
    str(DATA_ROOT),
    "--config",
    str(A1_PATH),
]
if A1_RESUME_DIR is not None:
    command += ["--resume-dir", str(A1_RESUME_DIR)]
run_stream(command)

## 7. Ver resultados e segurança de A1

In [ ]:
from PIL import Image
import pandas as pd

A1_OUT = (
    Path(A1_RESUME_DIR)
    if A1_RESUME_DIR
    else max(
        (PROJECT_ROOT / A1["runtime"]["output_root"]).iterdir(), key=lambda p: p.stat().st_mtime
    )
)
A1_SUM = json.loads((A1_OUT / "final_summary.json").read_text())
A1_SAFE = json.loads((A1_OUT / "augmentation_safety.json").read_text())
display(A1_SAFE)
display(Image.open(A1_OUT / "first_training_batch.png"))
display(Image.open(A1_OUT / "training_curves.png"))
display(Image.open(A1_OUT / "fixed_validation_panel.png"))

# Parte B — D0-A2

A2 começa do zero e não carrega pesos de A1.

In [ ]:
# Exemplo: A2_RESUME_DIR=PROJECT_ROOT/'results/day7/D0A2/D0A2_seed42_...'
A2_RESUME_DIR = None
command = [
    sys.executable,
    "scripts/run_augmentation_experiment.py",
    "--project-root",
    str(PROJECT_ROOT),
    "--data-root",
    str(DATA_ROOT),
    "--config",
    str(A2_PATH),
]
if A2_RESUME_DIR is not None:
    command += ["--resume-dir", str(A2_RESUME_DIR)]
run_stream(command)

## 8. Ver resultados e segurança de A2

In [ ]:
A2_OUT = (
    Path(A2_RESUME_DIR)
    if A2_RESUME_DIR
    else max(
        (PROJECT_ROOT / A2["runtime"]["output_root"]).iterdir(), key=lambda p: p.stat().st_mtime
    )
)
A2_SUM = json.loads((A2_OUT / "final_summary.json").read_text())
A2_SAFE = json.loads((A2_OUT / "augmentation_safety.json").read_text())
display(A2_SAFE)
display(Image.open(A2_OUT / "first_training_batch.png"))
display(Image.open(A2_OUT / "training_curves.png"))
display(Image.open(A2_OUT / "fixed_validation_panel.png"))

## 9. Tabela provisória somente da validação

Esta tabela organiza os resultados; a escolha ainda exigirá comparação pareada por imagem e incerteza.

In [ ]:
D0_DIR = PROJECT_ROOT / "results/day5/D0/D0_seed42_20260829T105601Z"
D0_SUM = json.loads((D0_DIR / "final_summary.json").read_text())
rows = []
for label, summary in [("A0/D0", D0_SUM), ("A1", A1_SUM), ("A2", A2_SUM)]:
    s = summary["selection"]
    rows.append(
        {
            "braço": label,
            "melhor_época": summary["training"]["best_epoch"],
            "Dice_0.5": s["metrics_at_0_5"]["macro"]["dice"]["mean"],
            "threshold": s["threshold"],
            "Dice_selecionado": s["metrics_selected"]["macro"]["dice"]["mean"],
            "IoU": s["metrics_selected"]["macro"]["iou"]["mean"],
            "precisão": s["metrics_selected"]["macro"]["precision"]["mean"],
            "recall": s["metrics_selected"]["macro"]["recall"]["mean"],
        }
    )
display(pd.DataFrame(rows))

## Artefatos da execução

Para cada braço, arquive `final_summary.json`, `history.csv`, `training_curves.png`, `threshold_curve.csv`, `threshold_curves.png`, `threshold_selection.json`, `fixed_validation_panel.png`, `validation_by_area_bin.csv`, `validation_per_image_selected.csv`, `augmentation_safety.json` e `first_training_batch.png`.

Não execute P1+A*, teste interno ou FUSeg antes da análise conjunta.